# P5 v1.4 — READ cache 및 probe
GPU 런타임에서 위에서 아래로 실행합니다. 세 동결 LM과 원본 초기 LM의 12층 h/u/m을 예약 READ 위치에서 추출합니다. cache 약 17.7GB, ZIP·재개 복사를 위해 로컬 여유 45GB와 Drive 여유 40GB 이상을 확보하세요. 완료한 행동 gate/test를 실행하지 않습니다.

1. 입력 ZIP 업로드·검증 → 2. 의존성 설치 → 3. Drive 연결/재개 → 4. 새 코드 GPU smoke 및 cache → 5. CPU probe → 6. 증빙 ZIP 저장·다운로드. 중단 시 동일 ZIP으로 처음부터 실행하면 checksum 완료 단위부터 재개합니다. Probe는 CPU 계산량이 크며 진행 결과를 단위별 저장합니다. cache만 먼저 반환해 로컬 CPU로 이어갈 수도 있습니다.

In [ ]:
from google.colab import files
from pathlib import Path
import hashlib, json, zipfile, os, shutil, subprocess, sys
uploaded=files.upload()
bundle=Path('v1_4_p5_bundle_r2.zip')
assert bundle.exists(), '지정된 P5 입력 ZIP을 업로드하세요.'
def sha(p):
    h=hashlib.sha256()
    with open(p,'rb') as f:
        for b in iter(lambda:f.read(1024*1024),b''): h.update(b)
    return h.hexdigest()
assert sha(bundle)=='afe16d82028fbefd603b601e17a68ab4719a83b1625d5f3099191c00d72991a3', '입력 ZIP checksum 불일치'
ROOT=Path('/content/boolean_interp');ROOT.mkdir(exist_ok=True)
with zipfile.ZipFile(bundle) as z:
    inventory=json.loads(z.read('bundle_manifest.json'))['files']
    assert set(z.namelist())==set(inventory)|{'bundle_manifest.json'}
    for name,d in inventory.items():
        dest=ROOT/name
        assert dest.resolve().is_relative_to(ROOT.resolve())
        if dest.exists(): assert sha(dest)==d, '기존 파일 충돌: '+name
        else: z.extract(name,ROOT)
        assert sha(dest)==d
os.chdir(ROOT)
del uploaded
print('입력 검증 완료')

## 의존성 설치
이전 실제 GPU 환경에서 저장한 주요 패키지 버전을 적용합니다. Colab 이미지가 달라지면 실행기가 새 환경 ID를 기록하고 현재 환경에서 smoke를 다시 수행합니다. 설치 후 런타임 재시작 안내가 나오면 재시작하고 셀을 다시 실행하세요.

In [ ]:
subprocess.run([sys.executable,'-m','pip','install','-r','experiment_v1_4/frozen_test_r1/requirements-primary.lock.txt','--extra-index-url','https://download.pytorch.org/whl/cu128'],check=True)
subprocess.run([sys.executable,'-m','pytest','tests_v1_4/test_p5.py','-q'],check=True)

## 영속 저장·재개
Drive의 전용 폴더를 사용합니다. 저장 파일을 수정하지 마세요. 기존 결과와 다른 계약은 자동으로 거부합니다. 새 런타임에서는 Drive의 완료 조각을 로컬로 복사하고 checksum을 검사합니다.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
PERSIST=Path('/content/drive/MyDrive/boolean_interp_v1_4/P5_r2')
OUT=ROOT/'experiment_v1_4/runs/p5_r2'
PERSIST.mkdir(parents=True,exist_ok=True);OUT.mkdir(parents=True,exist_ok=True)
for src in PERSIST.rglob('*'):
    if not src.is_file() or src.name.endswith('.tmp'):continue
    dest=OUT/src.relative_to(PERSIST);dest.parent.mkdir(parents=True,exist_ok=True)
    if dest.exists():assert sha(dest)==sha(src), '로컬/Drive 충돌: '+str(dest)
    else:
        shutil.copyfile(src,dest);assert sha(src)==sha(dest)
assert shutil.disk_usage(ROOT).free>25_000_000_000, '로컬 여유 공간을 확보하세요.'
SEEDS=[0,1,2] # 일부 seed 실행 후 같은 폴더에 나머지를 이어갈 수 있습니다.
def run(action):
    subprocess.run([sys.executable,'-m','interp_v1_4.p5',action,'--root',str(ROOT),'--output',str(OUT),
                    '--persistent',str(PERSIST),'--seeds',*map(str,SEEDS)],check=True)
print('재개 입력 복사 완료')

## GPU 추출
실행기가 현재 코드·환경의 debug smoke를 먼저 확인합니다. seed·split·checkpoint·READ 위치 hash가 같은 완전 저장 조각만 재사용합니다. GPU 메모리 부족 시 microbatch만 16→8→4→2→1로 줄입니다. quota·층·폭은 유지합니다.

In [ ]:
run('extract')

## CPU probe
이 셀은 CPU에서 실행합니다. 완료 JSON은 재사용하고 train/validation만으로 전처리·ANOVA·lambda·threshold를 선택합니다. test 점수는 보고에만 씁니다. 현재 GPU 런타임에서 CPU 작업을 이어가거나, cache를 보존한 뒤 CPU 런타임에서 앞 셀들의 입력/Drive 준비 후 이 셀부터 재개할 수 있습니다. GPU 추출 셀은 이미 끝난 경우 건너뜁니다. cache만 먼저 반환하려면 이 셀을 건너뛰고 다음 셀을 실행하세요.

In [ ]:
run('probes')

## 결과 보존·다운로드
cache와 probe를 함께 보존합니다. ZIP 생성 시 추가 로컬 공간이 필요합니다. 반환 검증에서 누락·실패·support 부족을 구분하므로 파일 생성만으로 P5를 완료 처리하지 않습니다. 대용량 ZIP 다운로드가 실패하면 Drive에 저장된 ZIP과 SHA256 파일을 내려받아 전달하세요.

In [ ]:
from datetime import datetime, timezone
stamp=datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%S')
archive=Path('/content')/f'v1_4_p5_evidence_{stamp}.zip'
subprocess.run([sys.executable,'-m','interp_v1_4.p5','export','--output',str(OUT),'--archive',str(archive)],check=True)
saved=PERSIST.parent/archive.name
shutil.copyfile(archive,saved);assert sha(archive)==sha(saved)
shutil.copyfile(str(archive)+'.sha256',str(saved)+'.sha256')
print('Drive 결과:',saved)
files.download(str(archive)+'.sha256')
files.download(str(archive))